In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda,RunnableBranch, RunnableParallel
from pydantic import BaseModel
from typing import Literal

In [33]:

# Base LLM

prompt_template = ChatPromptTemplate.from_messages([
    ("system","you are a movie review writter who analyse the ratings and review"),
    ("human","Write reivew with in 30 words for the movie : {input}")
])

llm = ChatOpenAI(model='gpt-5-mini', temperature=0)

class llm_schema(BaseModel):
    summary_flage: Literal['possitive','negative']

llm_structured_output = llm.with_structured_output(llm_schema)

In [ ]:

#Parellel Chain 1

fb_prompt= ChatPromptTemplate.from_messages([
    ("system","you are a serious content writter on Facebook"),
    ("human","create a serious post for the following text : {text}")
])

str_parser = StrOutputParser()

fb_chain = fb_prompt | llm | str_parser



In [40]:
from langchain_core.runnables import RunnableLambda

#Parellel Chain 2

def insta_chain(text):
    insta_prompt= ChatPromptTemplate.from_messages([
        ("system","you are a serious content writter on instagram"),
        ("human","create a serious post for the following text : {text}")
    ])

    str_parser = StrOutputParser()

    insta_chain = fb_prompt | llm | str_parser

    return insta_chain.invoke(text)

insta_chain_runnable = RunnableLambda(insta_chain)



In [19]:

# Conditional Chain

conditional_chain = RunnableBranch(
    
    (lambda x : x == 'possitive' , fb_chain),
    insta_chain
)

In [39]:
def dict_maker(text:dict):
    return text.model_dump()['summary_flage']

runnable_dict_maker = RunnableLambda(dict_maker)

In [34]:
test= prompt_template | llm_structured_output | runnable_dict_maker

In [35]:
res1 = test.invoke("Avatar 3")

In [36]:
res1

llm_schema(summary_flage='possitive')

In [38]:
res1.model_dump()['summary_flage']

'possitive'

In [41]:
#Final chain

final_chain = prompt_template | llm_structured_output | runnable_dict_maker | conditional_chain

In [44]:
response = final_chain.invoke("Avatar 3")

In [45]:
response

'Good morning, friends! 🌞 Today I’m choosing positivity — noticing the little wins, sharing a smile, and treating setbacks as lessons, not roadblocks. Let’s lift each other up: drop one good thing that happened to you this week in the comments and let’s celebrate it together! 💛 #ChoosePositive #Gratitude'

In [46]:
response2 = final_chain.invoke("Madame Web")

In [47]:
response2

'We all face negative moments — and that’s okay. What matters is how we respond: let those challenges teach you, strengthen you, and push you toward something better. Turn “I can’t” into “I’ll try,” reroute frustration into action, and celebrate the small wins along the way. Share one negative experience you turned into a lesson or victory — your story might inspire someone today. 🌱✨💪'